# Gaussian Process Regression from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/gaussian_process_regression.ipynb)

Build a Gaussian process regressor from scratch with NumPy. Learn how the kernel function encodes smoothness, how the posterior collapses with data, and why GPs are the engine behind Bayesian optimisation.

**Blog post:** [Gaussian Process Regression: The Bayesian Approach to Curve Fitting](https://sesen.ai/blog/gaussian-process-regression-from-scratch)

**Reference:** Ebden, M. (2008) ["Gaussian Processes for Regression: A Quick Introduction"](https://www.robots.ox.ac.uk/~mebden/reports/GPtutorial.pdf)

## 1. Setup

In [ ]:
import numpy as np
from numpy.linalg import inv, det
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

## 2. The Kernel Function

The squared exponential (RBF) kernel measures how correlated two function values are based on how close their inputs are:

$$k(x, x') = \sigma_f^2 \exp\!\left(-\frac{(x - x')^2}{2\ell^2}\right)$$

- $\sigma_f$ (signal std) — typical amplitude of the function
- $\ell$ (length-scale) — how far you need to move in $x$ before $f$ changes significantly
- $\sigma_n$ (noise std) — measurement noise

In [ ]:
def rbf_kernel(x1, x2, sigma_f, l):
    """Squared exponential (RBF) kernel."""
    return sigma_f**2 * np.exp(-0.5 * (x1 - x2)**2 / l**2)


def kernel_matrix(X1, X2, sigma_f, l):
    """Compute kernel matrix between two sets of points."""
    return np.array([[rbf_kernel(a, b, sigma_f, l) for b in X2] for a in X1])

## 3. GP Prediction

Given training data $(X, \mathbf{y})$ and test points $X_*$, the GP posterior is:

$$\bar{y}_* = K_* K^{-1} \mathbf{y}$$
$$\text{var}(y_*) = K_{**} - K_* K^{-1} K_*^\top$$

where $K$ is the training covariance, $K_*$ is test-train covariance, and $K_{**}$ is test-test covariance (Ebden Eq. 8 & 9).

In [ ]:
def gp_predict(X_train, y_train, X_test, sigma_f, l, sigma_n):
    """Predict mean and covariance at test points."""
    n = len(X_train)
    K = kernel_matrix(X_train, X_train, sigma_f, l) + sigma_n**2 * np.eye(n)
    K_s = kernel_matrix(X_test, X_train, sigma_f, l)
    K_ss = kernel_matrix(X_test, X_test, sigma_f, l) + sigma_n**2 * np.eye(len(X_test))

    K_inv = inv(K)
    mu = K_s @ K_inv @ y_train
    cov = K_ss - K_s @ K_inv @ K_s.T

    return mu, cov

## 4. Data from Ebden (2008)

Six noisy observations from the tutorial. The noise standard deviation $\sigma_n = 0.3$ is known from the error bars.

In [ ]:
# Training data from Ebden (2008) tutorial
X_train = np.array([-1.50, -1.00, -0.75, -0.40, -0.25, 0.00])
y_train = 0.55 * np.array([-3.0, -2.0, -0.6, 0.4, 1.0, 1.6])

# Hyperparameters (optimised values from the tutorial)
sigma_n = 0.3   # observation noise (known)
sigma_f = 1.27  # signal standard deviation (optimised)
l = 1.0         # length-scale (optimised)

print(f'Training data: {len(X_train)} points')
print(f'Hyperparameters: sigma_n={sigma_n}, sigma_f={sigma_f}, l={l}')

## 5. GP Posterior: Predict and Visualise

In [ ]:
# Predict on a dense grid
X_test = np.linspace(-2.0, 1.0, 200)
mu, cov = gp_predict(X_train, y_train, X_test, sigma_f, l, sigma_n)
std = np.sqrt(np.diag(cov))

# Plot the GP posterior
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(X_test, mu - 1.96 * std, mu + 1.96 * std,
                alpha=0.2, color='tab:blue', label='95% confidence')
ax.plot(X_test, mu, 'tab:blue', linewidth=2, label='GP mean')
ax.errorbar(X_train, y_train, yerr=sigma_n, fmt='ro', capsize=4,
            markersize=7, label='Training data', zorder=5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Gaussian Process Regression', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

# Verify prediction at x*=0.2 (Ebden's test point)
x_pt = np.array([0.2])
mu_pt, cov_pt = gp_predict(X_train, y_train, x_pt, sigma_f, l, sigma_n)
print(f'Prediction at x*=0.2: mean={mu_pt[0]:.2f}, var={cov_pt[0,0]:.2f}')
print(f'Ebden reference:      mean=0.95,  var=0.21')

## 6. Posterior Growth: Adding Observations One by One

Watch how the GP posterior sharpens as we add observations. The confidence band starts wide (prior) and collapses around each new data point.

In [ ]:
K_prior = kernel_matrix(X_test, X_test, sigma_f, l) + sigma_n**2 * np.eye(len(X_test))
prior_std = np.sqrt(np.diag(K_prior))

fig, ax = plt.subplots(figsize=(10, 5))

def update(frame_idx):
    ax.clear()
    ax.set_xlim(-2.0, 1.0)
    ax.set_ylim(-2.5, 2.0)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.grid(True, alpha=0.3)

    if frame_idx == 0:
        ax.fill_between(X_test, -1.96 * prior_std, 1.96 * prior_std,
                        alpha=0.2, color='tab:blue')
        ax.plot(X_test, np.zeros_like(X_test), 'tab:blue', linewidth=2)
        ax.set_title(f'GP Prior (0 observations)', fontsize=14)
    else:
        X_sub = X_train[:frame_idx]
        y_sub = y_train[:frame_idx]
        mu_sub, cov_sub = gp_predict(X_sub, y_sub, X_test, sigma_f, l, sigma_n)
        std_sub = np.sqrt(np.diag(cov_sub))

        ax.fill_between(X_test, mu_sub - 1.96 * std_sub, mu_sub + 1.96 * std_sub,
                        alpha=0.2, color='tab:blue')
        ax.plot(X_test, mu_sub, 'tab:blue', linewidth=2)
        ax.errorbar(X_sub, y_sub, yerr=sigma_n, fmt='ro', capsize=4,
                    markersize=7, zorder=5)
        ax.set_title(f'GP Posterior ({frame_idx} observation{"s" if frame_idx > 1 else ""})',
                    fontsize=14)
    return []

anim = FuncAnimation(fig, update, frames=len(X_train) + 1, interval=800, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 7. GP Prior: Sampling Random Functions

Before seeing data, the GP prior defines a distribution over functions. Every sample is smooth — the RBF kernel enforces this.

In [ ]:
X_grid = np.linspace(-2.0, 2.0, 200)
K_pr = kernel_matrix(X_grid, X_grid, sigma_f, l)
K_pr += 1e-8 * np.eye(len(X_grid))  # numerical stability

np.random.seed(42)
samples = np.random.multivariate_normal(np.zeros(len(X_grid)), K_pr, size=5)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(5):
    ax.plot(X_grid, samples[i], linewidth=1.5, alpha=0.8, label=f'Sample {i+1}')
ax.fill_between(X_grid, -1.96 * sigma_f, 1.96 * sigma_f, alpha=0.08,
                color='grey', label='95% prior band')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title(f'Random Functions from the GP Prior (l={l}, σ_f={sigma_f})', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.show()

## 8. Effect of Length-Scale

The length-scale $\ell$ controls the smoothness of the GP fit. Short $\ell$ gives wiggly functions; long $\ell$ gives over-smoothed ones.

In [ ]:
l_values = [0.3, 1.0, 3.0]
titles = ['Short (l=0.3) — Wiggly', 'Medium (l=1.0) — Balanced', 'Long (l=3.0) — Over-smoothed']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, l_val, title in zip(axes, l_values, titles):
    mu_l, cov_l = gp_predict(X_train, y_train, X_test, sigma_f, l_val, sigma_n)
    std_l = np.sqrt(np.diag(cov_l))

    ax.fill_between(X_test, mu_l - 1.96 * std_l, mu_l + 1.96 * std_l,
                    alpha=0.2, color='tab:blue')
    ax.plot(X_test, mu_l, 'tab:blue', linewidth=2)
    ax.errorbar(X_train, y_train, yerr=sigma_n, fmt='ro', capsize=3,
                markersize=5, zorder=5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('x', fontsize=11)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('y', fontsize=11)
fig.suptitle('Effect of Length-Scale on GP Regression', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Hyperparameter Optimisation via Marginal Log-Likelihood

The marginal log-likelihood (Ebden Eq. 10) balances data fit against model complexity:

$$\log p(\mathbf{y} \mid \mathbf{x}, \boldsymbol{\theta}) = -\frac{1}{2}\mathbf{y}^\top K^{-1}\mathbf{y} - \frac{1}{2}\log|K| - \frac{n}{2}\log 2\pi$$

In [ ]:
from scipy.optimize import minimize

def neg_log_marginal_likelihood(params, X_train, y_train, sigma_n):
    """Negative log marginal likelihood (Ebden Eq. 10)."""
    sigma_f, l = np.abs(params)  # ensure positive
    n = len(X_train)
    K = kernel_matrix(X_train, X_train, sigma_f, l) + sigma_n**2 * np.eye(n)

    log_lik = (-0.5 * y_train @ inv(K) @ y_train
               - 0.5 * np.log(det(K))
               - n / 2 * np.log(2 * np.pi))
    return -log_lik

# Optimise sigma_f and l (keep sigma_n=0.3 fixed, matching original code)
result = minimize(
    neg_log_marginal_likelihood,
    x0=[0.1, 0.5],  # initial values from original code
    args=(X_train, y_train, 0.3),
    method='Nelder-Mead'
)
sigma_f_opt, l_opt = np.abs(result.x)
print(f'Optimised: sigma_f={sigma_f_opt:.2f}, l={l_opt:.2f}')
print(f'Optimiser message: {result.message}')

### Log-Marginal Likelihood Landscape

In [ ]:
sf_range = np.linspace(0.2, 3.0, 60)
l_range = np.linspace(0.2, 3.0, 60)
SF, L = np.meshgrid(sf_range, l_range)
NLL = np.zeros_like(SF)

for i in range(len(l_range)):
    for j in range(len(sf_range)):
        NLL[i, j] = neg_log_marginal_likelihood(
            [SF[i, j], L[i, j]], X_train, y_train, sigma_n)

fig, ax = plt.subplots(figsize=(8, 6))
levels = np.linspace(np.nanmin(NLL), np.nanmin(NLL) + 15, 25)
cs = ax.contourf(SF, L, NLL, levels=levels, cmap='viridis_r')
plt.colorbar(cs, ax=ax, label='Negative Log-Likelihood')
ax.plot(1.27, 1.0, 'r*', markersize=15, markeredgecolor='white',
        markeredgewidth=1.5, label='Optimum (σ_f=1.27, l=1.0)')
ax.plot(0.1, 0.5, 'ws', markersize=8, markeredgecolor='red',
        markeredgewidth=1.5, label='Initial (σ_f=0.1, l=0.5)')
ax.set_xlabel('σ_f (signal std)', fontsize=12)
ax.set_ylabel('l (length-scale)', fontsize=12)
ax.set_title('Log-Marginal Likelihood Landscape', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
plt.show()

## 10. Comparison with scikit-learn

Verify our from-scratch implementation matches scikit-learn's `GaussianProcessRegressor`.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

kernel = 1.27**2 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.3**2)
gpr = GaussianProcessRegressor(kernel=kernel, optimizer=None, alpha=0)
gpr.fit(X_train.reshape(-1, 1), y_train)
mu_sk, std_sk = gpr.predict(X_test.reshape(-1, 1), return_std=True)

print(f'Max difference in mean: {np.max(np.abs(mu - mu_sk)):.2e}')
print(f'Max difference in std:  {np.max(np.abs(std - std_sk)):.2e}')

# Overlay both predictions
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(X_test, mu, 'tab:blue', linewidth=2, label='Our GP (NumPy)')
ax.plot(X_test, mu_sk, 'tab:orange', linewidth=2, linestyle='--', label='sklearn GPR')
ax.errorbar(X_train, y_train, yerr=sigma_n, fmt='ro', capsize=4, markersize=7, zorder=5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('NumPy vs sklearn: Identical Predictions', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

## Exercises

### Exercise 1: Matérn 5/2 Kernel

Implement the Matérn 5/2 kernel and compare its predictions to the RBF:

$$k_{\text{Matérn}}(r) = \sigma_f^2 \left(1 + \frac{\sqrt{5}r}{\ell} + \frac{5r^2}{3\ell^2}\right) \exp\!\left(-\frac{\sqrt{5}r}{\ell}\right), \quad r = |x - x'|$$

How do the confidence bands differ from the RBF?

### Exercise 2: Multi-Dimensional Inputs

Extend the GP to 2D inputs by modifying the kernel to use Euclidean distance:

$$k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

Generate 2D data from $f(x_1, x_2) = \sin(x_1) + \cos(x_2)$ and visualise the GP predictions as a surface.

### Exercise 3: Noisy Sinusoid

Generate 20 points from $y = \sin(x) + \epsilon$ where $\epsilon \sim \mathcal{N}(0, 0.2^2)$. Fit a GP and observe how the confidence band captures the noise.

### Exercise 4: Simple Bayesian Optimisation

Use your GP to implement Expected Improvement (EI) as an acquisition function. Iteratively select the next point to evaluate on $f(x) = -\sin(3x) - x^2 + 0.7x$ and find its minimum.